# nb-04 · Publish to CSV  *(stage: PUBLISH)*

**Purpose.** The tidy CSV is the deliverable. Finalize it: full processed audit,
the auto-generated column profile (the data dictionary's sanity check), and the
reconciliation scaffold against each agency's published totals.

| | |
|---|---|
| **Inputs** | `data/processed/reservoir-storage.csv` (from nb-03) |
| **Outputs** | `data/audit/summary-<ts>.md` · `docs/variables.csv` · `data/audit/reconcile.json` |
| **Preconditions** | nb-03 has run. |

A queryable **Datasette** surface + a **Quarto** docs site are deferred (Hub
roadmap) — this notebook stops at the CSV deliverable, as scoped in #9.

In [ ]:
# Make the `reservoir` package importable when running from notebooks/.
# Cleaner: `uv pip install -e ..` once, then this cell is a no-op.
import sys, pathlib
SRC = pathlib.Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('package path:', SRC)

In [ ]:
from reservoir import audit, config
from IPython.display import Markdown
Markdown(audit.audit_processed())

### Auto-generated column profile → `docs/variables.csv`
If this disagrees with `docs/data-dictionary.md`, the parser is usually wrong.

In [ ]:
audit.variables_report()

### Reconciliation (accuracy vs. the source's own published totals)

Fill `expected` with current storage figures read off each agency's
current-conditions page (see `docs/survey-notes.md`). Any `match == False` beyond
tolerance is a regression to investigate before publishing.

In [ ]:
expected = {
    # ('dwr_cdss', 'GREEN MOUNTAIN'): 138214.0,   # ← confirm from the DWR station page
}
audit.reconcile(expected) if expected else print('reconcile: fill expected totals (survey-notes)')

### The deliverable

In [ ]:
import pandas as pd
csv = config.CANONICAL_CSV
df = pd.read_csv(csv)
print('deliverable:', csv.relative_to(config.PROJECT_DIR))
print('rows:', len(df), '| columns:', list(df.columns))
print('load:  pd.read_csv(\'%s\', parse_dates=[\'datetime\'])' % csv.name)
print('wide views → docs/filter-pivot-recipes.md')